In [1]:
import time
import os
from tqdm import tqdm
import cv2
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
import numpy as np
import matplotlib.pyplot as plt

In [2]:
start_time = time.time()
# Folder path
folder_path = "Data_split/test/images"

# Create folder to save clustering results
output_folder_clustering = "Data_cluster_test_unet"
os.makedirs(output_folder_clustering, exist_ok=True)

# Create folder to save masking results
output_folder_masks = "Data_mask_test_unet"
os.makedirs(output_folder_masks, exist_ok=True)

# Create folder to save compared result of cluster images
output_compared = "Data_Compared_test_unet"
os.makedirs(output_compared, exist_ok=True)

# Create folder to save elbow metrics images
output_elbow = "Data_Elbow_test_unet"
os.makedirs(output_elbow, exist_ok=True)

# Sorted images
image_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".bmp")])

#selected_images = image_files[:5]

# Selected cluster for generate masks
selected_cluster = [0]

# Initialize tqdm outside the loop
with tqdm(total=len(image_files), desc="Processing images", unit="file") as pbar:
    for i, img_name in enumerate(image_files):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_BAYER_BGGR2RGB)
        assert img is not None, "File could not be read, check with os.path.exists()"

        # --- Flatten image ---
        image_2d = img.reshape((-1, 3))
        image_2d = np.float32(image_2d)

        # Model KMeans
        model = KMeans(random_state=42, n_init=10)

        # Using Elbow method to find optimal k
        visualizer = KElbowVisualizer(
            model, 
            k=(2, 12), 
            metric='distortion', 
            title=f"Distortion Score for KMeans Clustering Image {i}"
        )
        visualizer.fit(image_2d)
        
        # Save elbow plot
        output_path_elbow = os.path.join(output_elbow, f"Elbow_{img_name}.png")
        visualizer.show(outpath=output_path_elbow, clear_figure=True)

        # Take optimal k from the visualizer
        k_optimal = visualizer.elbow_value_
        print(f"Optimal number of clusters (k): {k_optimal}")

        # Convert the RGB image to HLS
        img_hls = cv2.cvtColor(img, cv2.COLOR_RGB2HLS)
        lightness = img_hls[:, :, 1]

        # Apply KMeans clustering
        kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
        labels = kmeans.fit_predict(image_2d)

        segmented_img = kmeans.cluster_centers_[labels].reshape(img.shape).astype(np.uint8)
        label_map = labels.reshape(img.shape[0], img.shape[1])

        # --- Sorting clusters by average lightness ---
        average_lightness = []
        for label in range(k_optimal):
            mask = (label_map == label)
            avg_l = np.mean(lightness[mask]) if np.any(mask) else 0
            average_lightness.append((label, avg_l))

        average_lightness.sort(key=lambda x: x[1])
        sorted_labels = {old_label: new_label for new_label, (old_label, _) in enumerate(average_lightness)}

        sorted_label_map = np.copy(label_map)
        for old_label, new_label in sorted_labels.items():
            sorted_label_map[label_map == old_label] = new_label

        sorted_segmented_img = np.zeros_like(segmented_img)
        for label, color in enumerate(kmeans.cluster_centers_.astype(np.uint8)):
            sorted_segmented_img[sorted_label_map == label] = color
       # --- Save clustered result in grayscale (0..255) ---
        sorted_label_map_gray = (sorted_label_map / (k_optimal - 1) * 255).astype(np.uint8)

        output_path_cluster = os.path.join(output_folder_clustering, f"Clustered_{img_name}")
        cv2.imwrite(output_path_cluster, sorted_label_map_gray)

        # --- Masking ---
        selected_cluster_mask = np.isin(sorted_label_map, selected_cluster)

        mask_image = np.zeros_like(img)
        mask_image[selected_cluster_mask] = [255, 255, 255]

        output_path_mask = os.path.join(output_folder_masks, f"Mask_{img_name}")
        cv2.imwrite(output_path_mask, cv2.cvtColor(mask_image, cv2.COLOR_RGB2BGR))

        # --- Plotting results ---
        plt.figure(figsize=(20, 6))

        plt.subplot(1, 5, 1)
        plt.imshow(img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1, 5, 2)
        plt.imshow(label_map, cmap="nipy_spectral")
        plt.title(f"Cluster (k={k_optimal})")
        plt.axis("off")

        plt.subplot(1, 5, 3)
        plt.imshow(sorted_label_map, cmap='gray')
        plt.title("Sorted by Lightness")
        plt.axis("off")

        plt.subplot(1, 5, 4)
        plt.imshow(sorted_label_map, cmap='nipy_spectral')
        plt.title("Clustered Result")
        plt.axis("off")

        plt.subplot(1, 5, 5)
        plt.imshow(mask_image)
        plt.title("Masking")
        plt.axis("off")

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.suptitle(f"Image {i}", fontsize=16, fontweight='bold')

        # Save plot
        output_path_compared = os.path.join(output_compared, f"Plot_{img_name}.png")
        plt.savefig(output_path_compared, dpi=300, bbox_inches='tight')

        plt.close()
        # Update progress bar
        pbar.update(1)

print(f"\nSegmentation done for all images!")

end_time = time.time()
execution_time = end_time - start_time
print(f"Total execution time: {execution_time:.2f} seconds")
print(f"Total execution time: {execution_time/60:.2f} minutes")


Processing images:   0%|          | 0/129 [00:00<?, ?file/s]

Optimal number of clusters (k): 4


Processing images:   1%|          | 1/129 [00:31<1:07:48, 31.79s/file]

Optimal number of clusters (k): 4


Processing images:   2%|▏         | 2/129 [01:03<1:06:55, 31.62s/file]

...

Processing images:  99%|█████████▉| 128/129 [1:04:06<00:29, 29.79s/file]

Optimal number of clusters (k): 4


Processing images: 100%|██████████| 129/129 [1:04:36<00:00, 30.05s/file]


Segmentation done for all images!
Total execution time: 3876.08 seconds
Total execution time: 64.60 minutes


<Figure size 800x550 with 0 Axes>